In [12]:
import requests
from bs4 import BeautifulSoup
import logging
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
import urllib.robotparser as robotparser


logger = logging.getLogger(__name__)


def make_session(user_agent: str = "paldat-scraper/1.0 (+https://example.org)",
                 retries: int = 3, backoff: float = 0.3, timeout: int = 10) -> requests.Session:
    s = requests.Session()
    s.headers.update({"User-Agent": user_agent})
    retry = Retry(total=retries, backoff_factor=backoff,
                  status_forcelist=(429, 500, 502, 503, 504), allowed_methods=("GET", "POST"))
    adapter = HTTPAdapter(max_retries=retry)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.request_timeout = timeout
    return s


def allowed_by_robots(base_url: str, user_agent: str, path: str) -> bool:
    rp = robotparser.RobotFileParser()
    rp.set_url(base_url.rstrip("/") + "/robots.txt")
    try:
        rp.read()
        return rp.can_fetch(user_agent, path)
    except Exception:
        # If robots cannot be fetched, default to being conservative elsewhere
        logger.debug("robots.txt could not be read; proceeding with caution")
        return True


def fetch(session: requests.Session, url: str) -> str:
    resp = session.get(url, timeout=getattr(session, "request_timeout", 10), verify=False)
    resp.raise_for_status()
    return resp.text

In [23]:
import json
import os
import scrape_utils

# r = requests.get('https://api.globalpollenproject.org/api/v1/Taxon/Sapindaceae')#, params={'id': '891fa5a1-ebfc-4360-83be-69fdf965219b/26.1.8v%20-%201'})
# response = r.json()
# for slide in response['Slides']:
#     print(slide)
#     url = f"https://globalpollenproject.org/Reference/{slide['ColId']}/{slide['SlideId'].replace(' ', '%20')}"
#     img = requests.get(url)
#     soup = BeautifulSoup(img.content, 'html.parser')
#     elements = soup.select('div .slide-gallery-item.col-md-3')
#     for el in elements:
#         print(el)
out_json = 'test.json'
os.makedirs('test', exist_ok=True)
saved = []
values_dict = json.load(open(out_json, 'r')) if os.path.exists(out_json) else {}
session = scrape_utils.make_session()
html = requests.get('https://api.globalpollenproject.org/api/v1/Taxon/Sapindaceae')    
for slide in html.json()['Slides']:
        id_counter = 1
        species_name = slide.get('LatinName', 'Unknown_Species')
        
        url = f"https://globalpollenproject.org/Reference/{slide['ColId']}/{slide['SlideId'].replace(' ', '%20')}"
        img = requests.get(url)
        soup = BeautifulSoup(img.content, 'html.parser')
        elements = soup.select('div .slide-gallery-item.col-md-3')
        # print(elements)
        for element in elements:
                img_url = element.get('data-frames', []).split(',')[0].replace('[', '').replace(']', '').replace('"', '').strip()
                print(type(img_url), img_url)
                
                img_data = session.get(img_url, timeout=getattr(session, 'request_timeout', 10), verify=False).content
                filename = f"{species_name}_{id_counter}.jpg"
                                       
                path = os.path.join('test', filename)
                with open(path, 'wb') as f:
                    f.write(img_data)
                saved.append(filename)
                id_counter += 1
                

<class 'str'> https://pollen.blob.core.windows.net/live/488bce8c-afdf-4e5a-9f39-d07ae1db9b88_0.png
<class 'str'> https://pollen.blob.core.windows.net/live/a50daa45-1b9e-44c2-87e2-e0eefacc0074_0.png
<class 'str'> https://pollen.blob.core.windows.net/live/ef66ef35-fd64-4d75-beab-0b1eedd55c9d_0.png
<class 'str'> https://pollen.blob.core.windows.net/live/cc149563-2752-41a5-a50d-96f25984fac2_0.png
<class 'str'> https://pollen.blob.core.windows.net/live/cb04594d-7bbe-49a6-9e91-a56a177f6ed6_0.png
<class 'str'> https://pollen.blob.core.windows.net/live/8b1f1dba-b56f-4920-90ba-70e32fbd5b27_0.png
<class 'str'> https://pollen.blob.core.windows.net/live/b41af788-d02e-4c83-815b-8826b352d004_0.png
<class 'str'> https://pollen.blob.core.windows.net/live/2e299e6f-0dd7-43d1-874f-6891f7a30895_0.png
<class 'str'> https://pollen.blob.core.windows.net/live/a6893ac6-d2e3-4d11-aeba-85d659cbbd5a_0.png
<class 'str'> https://pollen.blob.core.windows.net/live/68db30ab-ca72-435e-8e67-49950aa91515_0.png
<class 'st

In [ ]:

import json
import logging
import os
import random
import time

import requests
requests.packages.urllib3.disable_warnings()
from bs4 import BeautifulSoup

import scrape_utils

logger = logging.getLogger(__name__)

def download_images(link: str, outdir: str, id_counter: int, out_json: str = 'C:/Users/pinto/py_stuff/pollen-main/pollen-main/Web Scraping/gpp_out.json'):
    os.makedirs(outdir, exist_ok=True)
    
    
    if os.path.exists(out_json):
        with open(out_json, 'r') as f:
            values_dict = json.load(f) 
    else:
        values_dict = {}
    session = scrape_utils.make_session()
    html = requests.get(f'https://api.globalpollenproject.org/api/v1{link}')
    #rint(values_dict)
    for slide in html.json()['Slides']:
        print(slide)
        species_name = slide.get('LatinName', 'Unknown_Species')
        saved = []
        url = f"https://globalpollenproject.org/Reference/{slide['ColId']}/{slide['SlideId'].replace(' ', '%20')}"
        img = requests.get(url)
        soup = BeautifulSoup(img.content, 'html.parser')
        elements = soup.select('div .slide-gallery-item.col-md-3')
        for element in elements:
                try:    
                    img_url = element.get('data-frames', []).split(',')[0].replace('[', '').replace(']', '').replace('"', '').strip()
                    print(type(img_url), img_url)
                    
                    img_data = session.get(img_url, timeout=getattr(session, 'request_timeout', 10), verify=False).content

                    filename = f"{species_name}_{id_counter}.jpg"
                                        
                    path = os.path.join('GPP_images', filename)
                    with open(path, 'wb') as f:
                        f.write(img_data)
                    saved.append(filename)
                    id_counter += 1
                except Exception:
                    logger.exception('failed to download images for species %s', species_name)
        values_dict[species_name] = saved
        time.sleep(random.uniform(1.0, 3.0))
        logger.info('finished: %d images saved', len(saved))
        logger.info('finished: %d species saved', len(values_dict))
    #rint(values_dict)
    if os.path.exists(out_json):
        with open(out_json, 'a') as f:
            json.dump(values_dict, f)
    else:
        with open(out_json, 'w') as f:
            json.dump(values_dict, f)
download_images('/Taxon/Sapindaceae', 'GPP_images', 1)

{'ColId': 'de35675e-55b9-4798-a3f0-e7a57c174c8a', 'SlideId': 'JH Whirter 130382', 'LatinName': 'Molinaea tolambitou', 'Rank': 'Species', 'Thumbnail': 'https://pollen.blob.core.windows.net/live-cache/c4aa9701-031e-4cf9-a5f4-b8b31c567d9e_2_thumb.jpg'}
<class 'str'> https://pollen.blob.core.windows.net/live/488bce8c-afdf-4e5a-9f39-d07ae1db9b88_0.png
<class 'str'> https://pollen.blob.core.windows.net/live/a50daa45-1b9e-44c2-87e2-e0eefacc0074_0.png
{'ColId': 'de35675e-55b9-4798-a3f0-e7a57c174c8a', 'SlideId': 'GPP395', 'LatinName': 'Stadmania oppositifolia', 'Rank': 'Species', 'Thumbnail': 'https://pollen.blob.core.windows.net/live-cache/5768a856-cd8d-4cfd-a5aa-2bc306665c9e_2_thumb.jpg'}
<class 'str'> https://pollen.blob.core.windows.net/live/ef66ef35-fd64-4d75-beab-0b1eedd55c9d_0.png
{'ColId': 'de35675e-55b9-4798-a3f0-e7a57c174c8a', 'SlideId': 'GPP394', 'LatinName': 'Radlkofera calodendron', 'Rank': 'Species', 'Thumbnail': 'https://pollen.blob.core.windows.net/live-cache/5b835615-a205-4722-

TypeError: JSONEncoder.__init__() got an unexpected keyword argument 'encoding'

In [48]:
session = scrape_utils.make_session()
html = requests.get('https://api.globalpollenproject.org/api/v1/Taxon/Achariaceae', timeout=getattr(session, 'request_timeout', 10), verify=False)
print(html.json())    
#family_name = html.json()['Family']

{'Id': '35eed32f-74c3-445d-af65-cb63b55c3def', 'Family': 'Achariaceae', 'Genus': '', 'Species': '', 'LatinName': 'Achariaceae', 'Authorship': '', 'Rank': 'Family', 'Parent': None, 'Children': [{'Id': '84e30ccd-c691-4357-b1b9-d0b62f5f27ba', 'Name': 'Caloncoba', 'Rank': 'Genus'}, {'Id': 'a10f4797-9ff3-4855-a936-ffe59cb689e6', 'Name': 'Buchnerodendron', 'Rank': 'Genus'}, {'Id': '7ab530d9-ae74-470e-8a14-d1b478d3aadd', 'Name': 'Scottellia', 'Rank': 'Genus'}], 'Slides': [{'ColId': 'de35675e-55b9-4798-a3f0-e7a57c174c8a', 'SlideId': 'GPP169', 'LatinName': 'Caloncoba glauca', 'Rank': 'Species', 'Thumbnail': 'https://pollen.blob.core.windows.net/live-cache/fbefed9c-5ea7-4a15-b021-324318ebb0f6_2_thumb.jpg'}, {'ColId': 'de35675e-55b9-4798-a3f0-e7a57c174c8a', 'SlideId': 'GPP168', 'LatinName': 'Buchnerodendron speciosum', 'Rank': 'Species', 'Thumbnail': 'https://pollen.blob.core.windows.net/live-cache/8e2098ce-8ac2-4be0-9545-b9ae6e805518_2_thumb.jpg'}, {'ColId': 'de35675e-55b9-4798-a3f0-e7a57c174c8a

In [57]:
test = {'Acanthus spinosus': {'images': ['Acanthus spinosus_0.jpg', 'Acanthus spinosus_1.jpg', 'Acanthus spinosus_2.jpg'], 'family': 'Acanthaceae'}, 'Justicia elegantula': {'images': ['Justicia elegantula_3.jpg', 'Justicia elegantula_4.jpg', 'Justicia elegantula_5.jpg'], 'family': 'Acanthaceae'}, 'Justicia extensa': {'images': ['Justicia extensa_6.jpg', 'Justicia extensa_7.jpg'], 'family': 'Acanthaceae'}, 'Hypoestes verticillaris': {'images': ['Hypoestes verticillaris_8.jpg', 'Hypoestes verticillaris_9.jpg'], 'family': 'Acanthaceae'}, 'Blepharis subvolubilis': {'images': ['Blepharis subvolubilis_10.jpg', 'Blepharis subvolubilis_11.jpg'], 'family': 'Acanthaceae'}, 'Asystasia vogeliana': {'images': ['Asystasia vogeliana_12.jpg', 'Asystasia vogeliana_13.jpg'], 'family': 'Acanthaceae'}, 'Asystasia mysorensis': {'images': ['Asystasia mysorensis_14.jpg', 'Asystasia mysorensis_15.jpg'], 'family': 'Acanthaceae'}, 'Whitfieldia elongata': {'images': ['Whitfieldia elongata_16.jpg'], 'family': 'Acanthaceae'}, 'Asystasia gangetica': {'images': ['Asystasia gangetica_17.jpg', 'Asystasia gangetica_18.jpg', 'Asystasia gangetica_19.jpg'], 'family': 'Acanthaceae'}, 'Barleria kirkii': {'images': ['Barleria kirkii_20.jpg', 'Barleria kirkii_21.jpg'], 'family': 'Acanthaceae'}, 'Avicennia germinans': {'images': ['Avicennia germinans_22.jpg'], 'family': 'Acanthaceae'}, 'Ruellia jaliscana': {'images': ['Ruellia jaliscana_23.jpg'], 'family': 'Acanthaceae'}, 'Justicia hyssopifolia': {'images': ['Justicia hyssopifolia_24.jpg'], 'family': 'Acanthaceae'}, 'Justicia galapagana': {'images': ['Justicia galapagana_25.jpg', 'Justicia galapagana_26.jpg', 'Justicia galapagana_27.jpg'], 'family': 'Acanthaceae'}, 'Acanthus mollis': {'images': ['Acanthus mollis_28.jpg'], 'family': 'Acanthaceae'}}
out_json: str = 'C:\\Users\\pinto\\py_stuff\\pollen-main\\pollen-main\\Web Scraping\\output\\gpp_out.json'

if os.path.exists(out_json):
        #logger.info('json path exists %s', values_dict)
        with open(out_json, 'w') as f:
            json.dump(test, f, indent=2)
else:
        #logger.info('writing new json with %s species', values_dict)
        with open(out_json, 'w') as f:
            json.dump(test, f, indent=2)

In [58]:
if os.path.exists(out_json):
        with open(out_json, 'r') as f:
            values_dict = json.load(f) 
values_dict

{'Acanthus spinosus': {'images': ['Acanthus spinosus_0.jpg',
   'Acanthus spinosus_1.jpg',
   'Acanthus spinosus_2.jpg'],
  'family': 'Acanthaceae'},
 'Justicia elegantula': {'images': ['Justicia elegantula_3.jpg',
   'Justicia elegantula_4.jpg',
   'Justicia elegantula_5.jpg'],
  'family': 'Acanthaceae'},
 'Justicia extensa': {'images': ['Justicia extensa_6.jpg',
   'Justicia extensa_7.jpg'],
  'family': 'Acanthaceae'},
 'Hypoestes verticillaris': {'images': ['Hypoestes verticillaris_8.jpg',
   'Hypoestes verticillaris_9.jpg'],
  'family': 'Acanthaceae'},
 'Blepharis subvolubilis': {'images': ['Blepharis subvolubilis_10.jpg',
   'Blepharis subvolubilis_11.jpg'],
  'family': 'Acanthaceae'},
 'Asystasia vogeliana': {'images': ['Asystasia vogeliana_12.jpg',
   'Asystasia vogeliana_13.jpg'],
  'family': 'Acanthaceae'},
 'Asystasia mysorensis': {'images': ['Asystasia mysorensis_14.jpg',
   'Asystasia mysorensis_15.jpg'],
  'family': 'Acanthaceae'},
 'Whitfieldia elongata': {'images': ['Whi